# Extract ERA5-Land Weather Features

This notebook extracts the **weather reanalysis** features for the pipeline from
`ECMWF/ERA5_LAND/HOURLY` on Google Earth Engine and aligns them to the same
**1 km, daily, EPSG:4326 grid** as the other sources. Its output feeds
`0_4_merge_files.ipynb` as `weatherData-YYYY-MM.parquet`.

**Features exported** (must match the downstream `NUMERIC_FEATURES`):
`temperature_2m`, `dewpoint_temperature_2m`, `soil_temperature_level_1`,
`surface_net_thermal_radiation`, `u_component_of_wind_10m`,
`v_component_of_wind_10m`, `surface_pressure`, `total_precipitation`.

## Hourly → daily aggregation
ERA5-Land is hourly. Following the pipeline's convention, each day is aggregated
per band:
- **sum** → `total_precipitation`
- **max** → `u_component_of_wind_10m`, `v_component_of_wind_10m` (peak-wind proxy)
- **mean** → all remaining bands

> ⚠️ **ERA5-Land accumulation caveat.** In `ECMWF/ERA5_LAND/HOURLY`, accumulated
> bands (e.g. `total_precipitation`, `surface_net_thermal_radiation`) are
> accumulations **since 00 UTC**, so a day's total is the value at the *last* hour
> rather than the sum of the hourly steps. This notebook uses the aggregation the
> project documented (sum for precipitation). If your precipitation/radiation
> values look inflated, switch those bands to `.max()` (≈ last hour of day) or use
> the ready-made daily product `ECMWF/ERA5_LAND/DAILY_AGGR`. Verify against how you
> exported weather originally before merging.

In [ ]:
from google.colab import drive
import os
import ee
import time

# ---------------------------
# 1. Mount Google Drive
# ---------------------------
drive.mount('/content/drive', force_remount=True)

# Base export folder
export_folder = '/content/drive/My Drive/GEE_Exports/'
dataset_name = "ECMWF/ERA5_LAND/HOURLY"
epsm = 1000  # 1000m grid, same as the other sources

# ERA5-Land weather features (must match the downstream NUMERIC_FEATURES)
attributes = [
    'temperature_2m', 'dewpoint_temperature_2m', 'soil_temperature_level_1',
    'surface_net_thermal_radiation', 'u_component_of_wind_10m',
    'v_component_of_wind_10m', 'surface_pressure', 'total_precipitation',
]

# Per-band hourly -> daily aggregation (see the note above)
SUM_BANDS  = ['total_precipitation']                                 # daily accumulation
MAX_BANDS  = ['u_component_of_wind_10m', 'v_component_of_wind_10m']  # peak-wind proxy
MEAN_BANDS = ['temperature_2m', 'dewpoint_temperature_2m', 'soil_temperature_level_1',
              'surface_net_thermal_radiation', 'surface_pressure']

# Define province and year variables
country = 'Canada'
province = "Alberta"
year = "2024"

# List of months to process with their start and end dates
months_to_run = {
    "07": (f'{year}-07-01', f'{year}-07-31'),
    "08": (f'{year}-08-01', f'{year}-08-31'),
}

# ---------------------------
# 2. Initialize Earth Engine
# ---------------------------
cloud_project = 'ee-1wangjas2'

try:
    ee.Initialize(project=cloud_project)
    print('Earth Engine initialized successfully.')
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)
    print('Earth Engine initialized after authentication.')

# Load level 2 administrative boundaries
gaul_cities = ee.FeatureCollection('FAO/GAUL/2015/level2')
canada_cities = gaul_cities.filter(ee.Filter.eq('ADM0_NAME', country))
province_cities = canada_cities.filter(ee.Filter.eq('ADM1_NAME', province))

# ---------------------------
# Helper Functions (same conventions as 0_1 / 0_3)
# ---------------------------
def estimate_computation_complexity(geometry, scale):
    area = geometry.area(1).getInfo()
    return area / (scale * scale)

def get_optimal_tile_scale(grid_count):
    if grid_count > 1000000:
        return 2
    elif grid_count > 100000:
        return 4
    elif grid_count > 10000:
        return 8
    else:
        return 16

def zonal_stats_with_date(feature_collection, image, tile_scale):
    stats = image.reduceRegions(
        collection=feature_collection,
        reducer=ee.Reducer.mean(),
        scale=epsm,
        tileScale=tile_scale
    )
    return stats.map(lambda f: f.set({
        'longitude': f.geometry().centroid(1).coordinates().get(0),
        'latitude':  f.geometry().centroid(1).coordinates().get(1),
        'date':      ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    }))

def daily_composite(day, region_geometry):
    """Aggregate ERA5-Land HOURLY into one daily image with per-band reducers."""
    start = day
    end = day.advance(1, 'day')
    hourly = (ee.ImageCollection(dataset_name)
                .filterDate(start, end)
                .filterBounds(region_geometry))
    mean_img = hourly.select(MEAN_BANDS).mean()
    max_img  = hourly.select(MAX_BANDS).max()
    sum_img  = hourly.select(SUM_BANDS).sum()
    comp = mean_img.addBands(max_img).addBands(sum_img).select(attributes)
    return (comp.set('system:time_start', day.millis())
                .set('date', day.format('YYYY-MM-dd')))

def process_city(feature, start_date, end_date):
    city_name = feature.get('ADM2_NAME').getInfo().replace(" ", "_")
    region_geometry = feature.geometry()

    grid_count = estimate_computation_complexity(region_geometry, epsm)
    tile_scale = get_optimal_tile_scale(grid_count)

    projection = ee.Projection('EPSG:4326').atScale(epsm)
    grids = region_geometry.coveringGrid(projection)

    # Build one daily composite image per day in the month
    start = ee.Date(start_date)
    end = ee.Date(end_date).advance(1, 'day')
    num_days = end.difference(start, 'day')
    days = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start.advance(d, 'day'))
    comps = ee.ImageCollection(days.map(lambda d: daily_composite(ee.Date(d), region_geometry)))

    per_image_stats = comps.map(lambda image: zonal_stats_with_date(grids, image, tile_scale))
    all_stats = ee.FeatureCollection(per_image_stats.flatten())

    # Export to Google Drive (same layout as the other 0_* notebooks)
    export_task = ee.batch.Export.table.toDrive(
        collection=all_stats,
        description=f'{city_name}_ERA5_{start_date}',
        folder=f'ERA5/{start_date[:7]}',
        fileNamePrefix=f'{city_name}_ERA5_{start_date}',
        fileFormat='CSV',
        selectors=['date', 'longitude', 'latitude'] + attributes
    )
    export_task.start()
    print(f"Export task for {city_name} started with tileScale {tile_scale}")

# ---------------------------
# Main Processing Loop (mirrors 0_1_surface_temperature)
# ---------------------------
for month, (start_date, end_date) in months_to_run.items():
    province_folder = os.path.join(export_folder, province)
    os.makedirs(province_folder, exist_ok=True)

    year_folder = os.path.join(province_folder, year)
    os.makedirs(year_folder, exist_ok=True)

    month_folder = os.path.join(year_folder, month)
    os.makedirs(month_folder, exist_ok=True)

    dataset_folder = os.path.join(month_folder, "ERA5")
    os.makedirs(dataset_folder, exist_ok=True)

    os.chdir(dataset_folder)
    print(f'Current Directory: {os.getcwd()}')

    province_city_list = province_cities.toList(province_cities.size())
    city_count = province_cities.size().getInfo()
    print(f"Processing {city_count} cities in {province} for {month}-{year}")

    for i in range(city_count):
        city_feature = ee.Feature(province_city_list.get(i))
        process_city(city_feature, start_date, end_date)

print("All export tasks initiated. Check the Earth Engine Tasks tab for progress.")

## (Optional) Consolidate CSVs → `weatherData-YYYY-MM.parquet`

The GEE export above writes one CSV per administrative division to Drive (same as
the other `0_*` notebooks). Run the cell below to concatenate a month's CSVs into
the single parquet that `0_4_merge_files.ipynb` expects.

In [ ]:
# OPTIONAL: consolidate the exported per-division ERA5 CSVs for one month into a
# single parquet named the way 0_4_merge_files.ipynb expects.
import glob
import pandas as pd

YEAR_MONTH = "2024-07"  # <-- set to the month you exported above

csv_dir  = f'/content/drive/My Drive/GEE_Exports/ERA5/{YEAR_MONTH}'
out_path = f'/content/drive/My Drive/GEE_Exports/{province}/{year}/ERA5/weatherData-{YEAR_MONTH}.parquet'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(csv_dir, '*.csv')))
print(f"Found {len(csv_files)} CSV files in {csv_dir}")

frames = [pd.read_csv(f) for f in csv_files]
df = pd.concat(frames, ignore_index=True)
df['date'] = pd.to_datetime(df['date'])
df = df.drop_duplicates(subset=['date', 'latitude', 'longitude'])

df.to_parquet(out_path, index=False)
print(f"Wrote {len(df)} rows -> {out_path}")